In [1]:
import pandas as pd
import numpy as np

master_df = pd.read_csv("../outputs/master_customer_table.csv")

master_df.head()

,customer_unique_id,order_id,order_purchase_timestamp,order_status,product_id,price,review_score,review_comment_message,product_category_name
0,861eff4711a542e4b93843c6dd7febb0,00e7ee1b050b8499577073aeb2a297a1,2017-05-16 15:05:35,delivered,a9516a079e37a9c9c36b9b78b10169e8,124.99,4.0,NaN,moveis_escritorio
1,290c77bc529b7ac935b93aa66c333dc3,29150127e6685892b6eab3eec79f59c7,2018-01-12 20:48:24,delivered,4aa6014eceb682077f9dc4bffebc05b0,289.00,5.0,NaN,utilidades_domesticas
2,060e732b5b29e8181a18229c7b0b2b5e,b2059ed67ce144a36e2aa97d2c9e9ad2,2018-05-19 16:07:45,delivered,bd07b66896d6f1494f5b86251848ced7,139.94,5.0,NaN,moveis_escritorio
3,259dac757896d24d7702b9acbbff3f3c,951670f92359f4fe4a63112aa7306eba,2018-03-13 16:06:38,delivered,a5647c44af977b148e0a3a4751a09e2e,149.94,5.0,NaN,moveis_escritorio
4,345ecd01c38d18a9036ed96c73b8d066,6b7d50bd145f6fc7f33cebabd7e49d0f,2018-07-29 09:51:30,delivered,9391a573abe00141c56e38d84d7d5b3b,230.00,5.0,O baratheon è esxelente Amo adoro o baratheon,casa_conforto


In [2]:
# Keep only rows that have review text
reviews_df = master_df.dropna(subset=["review_comment_message"])

print("Rows with review text:", reviews_df.shape)
reviews_df.head()

Rows with review text: (47642, 9)


,customer_unique_id,order_id,order_purchase_timestamp,order_status,product_id,price,review_score,review_comment_message,product_category_name
4,345ecd01c38d18a9036ed96c73b8d066,6b7d50bd145f6fc7f33cebabd7e49d0f,2018-07-29 09:51:30,delivered,9391a573abe00141c56e38d84d7d5b3b,230.0,5.0,O baratheon è esxelente Amo adoro o baratheon,casa_conforto
5,4c93744516667ad3b8f1fb645a3116a4,5741ea1f91b5fbab2bd2dc653a5b5099,2017-09-14 18:14:31,delivered,0be701e03657109a8a4d5168122777fb,259.9,5.0,Loja responsável,esporte_lazer
8,1175e95fb47ddff9de6b2b06188f7e0d,1ebeea841c590e86a14a0d7a48e7d062,2018-01-18 12:35:44,delivered,08401f48b67a852d06105725c3013eff,25.0,5.0,chegou antes do prometido,brinquedos
9,9afe194fb833f79e300e37e580171f22,7433cbcc783205509d66a5260da5b574,2018-01-08 11:22:34,delivered,78efe838c04bbc568be034082200ac20,99.9,5.0,"ótimo, entregou antes da data prevista.",moveis_decoracao
10,2a7745e1ed516b289ed9b29c7d0539a5,8428e578bb1cf839ae26a6b7615502b9,2017-11-27 17:23:20,delivered,6b874ff4c71d18e91079b40396bf176a,25.3,5.0,Td certo.Produto e prazo de entrega.,automotivo


In [3]:
customer_reviews = (
    reviews_df
    .groupby("customer_unique_id")["review_comment_message"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

print(customer_reviews.shape)
customer_reviews.head()

(39322, 2)


,customer_unique_id,review_comment_message
0,0000366f3b9a7992bf8c76cfdf3221e2,"Adorei a cortina, ficou linda na minha sala, e..."
1,0000f6ccb0745a6a4b88665a16c9f078,Bom vendedor
2,00053a61a98854899e70ed204dd4bafe,Olá! Comprei dois potes de whey e chegou apena...
3,0005ef4cd20d2893f0d9fbd94d3c0d97,Até o presente momento não recebi o produto e ...
4,0006fdc98a402fceb4eb0ee528f6a8d4,só achei que a altura da saia poderia ser maior


In [4]:
from sentence_transformers import SentenceTransformer

# Force CPU to avoid GPU/CUDA issues
model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cuda"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
embeddings = model.encode(
    customer_reviews["review_comment_message"].tolist(),
    batch_size=16,        # DO NOT increase beyond 16
    show_progress_bar=True
)

Batches:   0%|          | 0/2458 [00:00<?, ?it/s]

In [7]:
embeddings.shape

(39322, 384)

In [9]:
import umap

umap_model = umap.UMAP(
    n_components=2,
    random_state=42
)

embeddings_2d = umap_model.fit_transform(embeddings)
embeddings_2d.shape

C:\Users\gujar\anaconda3\envs\torch_gpu\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\gujar\anaconda3\envs\torch_gpu\lib\site-packages\umap\spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(


(39322, 2)

In [10]:
nlp_features = pd.DataFrame(
    embeddings_2d,
    columns=["nlp_1", "nlp_2"]
)

nlp_features["customer_unique_id"] = customer_reviews["customer_unique_id"]

nlp_features.head()

,nlp_1,nlp_2,customer_unique_id
0,7.955969,0.534592,0000366f3b9a7992bf8c76cfdf3221e2
1,13.129206,11.164166,0000f6ccb0745a6a4b88665a16c9f078
2,4.529377,4.633020,00053a61a98854899e70ed204dd4bafe
3,5.494314,1.528532,0005ef4cd20d2893f0d9fbd94d3c0d97
4,5.508366,-0.480954,0006fdc98a402fceb4eb0ee528f6a8d4


In [11]:
nlp_features.to_csv(
    "../outputs/nlp_features.csv",
    index=False
)

print("nlp_features.csv saved successfully.")

nlp_features.csv saved successfully.
